In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА 1: УСТАНОВКА И ПРОВЕРКА ВСЕХ МОДЕЛЕЙ ПЕРЕД ОБУЧЕНИЕМ
# ──────────────────────────────────────────────────────────────────────────────

# Установка библиотек (раскомментируйте при первом запуске)
# !pip install -q transformers==4.40.0 datasets accelerate huggingface_hub

import gc
import torch
import json
import pandas as pd
from huggingface_hub import HfApi

# Очищаем память
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("="*70)
print("ШАГ 1: ПРОВЕРКА ДОСТУПНОСТИ МОДЕЛЕЙ НА HUGGING FACE")
print("="*70)

def check_model_exists(model_name):
    """
    Проверяет существование модели на Hugging Face Hub
    """
    try:
        api = HfApi()
        model_info = api.model_info(model_name)
        # Дополнительная проверка: можно ли загрузить конфиг?
        if hasattr(model_info, 'siblings'):
            config_exists = any('config.json' in s.rfilename for s in model_info.siblings)
            if not config_exists:
                return False, "Нет config.json"
        print(f"  ✓ {model_name}")
        return True, model_info
    except Exception as e:
        error_msg = str(e)
        if "401" in error_msg or "403" in error_msg:
            print(f"  ✗ {model_name} — требуется авторизация")
        elif "404" in error_msg or "not found" in error_msg.lower():
            print(f"  ✗ {model_name} — не найдена")
        else:
            print(f"  ✗ {model_name} — {error_msg[:80]}")
        return False, None

# Список моделей для проверки (все потенциально рабочие)
CANDIDATE_MODELS = {
    # Основные модели (должны работать)
    "primary": [
        "deepset/deberta-v3-base-squad2",
        "deepset/roberta-base-squad2",
        "deepset/roberta-large-squad2",
        "deepset/electra-base-squad2",
    ],
    # CUAD-специализированные модели (разные авторы)
    "cuad": [
        "mgigena/roberta-large-cuad",
        "akdeniz27/roberta-large-cuad",
        "Yanzhu/roberta-large-cuad",
        "Narrativa/roberta-large-cuad",
    ],
    # Запасные варианты (если основные недоступны)
    "fallback": [
        "distilbert-base-cased-distilled-squad",
        "bert-large-uncased-whole-word-masking-finetuned-squad",
        "albert-xxlarge-v2-squad2",
    ]
}

# Проверяем все модели
available_models = {
    "primary": [],
    "cuad": [],
    "fallback": []
}

print("\nПроверка основных моделей (SQuAD2.0):")
for model in CANDIDATE_MODELS["primary"]:
    exists, _ = check_model_exists(model)
    if exists:
        available_models["primary"].append(model)

print("\nПроверка CUAD-специализированных моделей:")
for model in CANDIDATE_MODELS["cuad"]:
    exists, _ = check_model_exists(model)
    if exists:
        available_models["cuad"].append(model)

# Если нет ни одной CUAD-модели, используем fallback
if not available_models["cuad"]:
    print("\n⚠️ CUAD-модели не найдены, проверяем запасные варианты:")
    for model in CANDIDATE_MODELS["fallback"]:
        exists, _ = check_model_exists(model)
        if exists:
            available_models["fallback"].append(model)

# Формируем финальный список моделей (максимум 5, приоритет CUAD)
final_models = []

# Сначала добавляем CUAD-модели (до 2 штук)
final_models.extend(available_models["cuad"][:2])

# Затем основные SQuAD модели (до 3 штук)
final_models.extend(available_models["primary"][:3])

# Если всё ещё мало, добавляем fallback
if len(final_models) < 3:
    needed = 3 - len(final_models)
    final_models.extend(available_models["fallback"][:needed])

print("\n" + "="*70)
print("РЕЗУЛЬТАТ ПРОВЕРКИ:")
print(f"  Найдено CUAD-моделей: {len(available_models['cuad'])}")
print(f"  Найдено SQuAD-моделей: {len(available_models['primary'])}")
print(f"  Найдено fallback-моделей: {len(available_models['fallback'])}")
print(f"  Итого в ансамбле: {len(final_models)}/5 моделей")
print("="*70)

if len(final_models) < 2:
    print("\n❌ КРИТИЧЕСКАЯ ОШИБКА: Найдено менее 2 моделей!")
    print("  Возможные причины:")
    print("  1. Нет интернет-соединения")
    print("  2. Блокировка доступа к HuggingFace")
    print("  3. Требуется авторизация (выполните: huggingface-cli login)")
    print("\n  Попробуйте использовать локальные модели или проверьте соединение.")
    raise RuntimeError("Недостаточно моделей для ансамбля")
else:
    print(f"\n✓ Успешно! Будет использовано {len(final_models)} моделей:")

# Конфигурация для каждой модели (веса, гиперпараметры)
MODELS_CONFIG = []

# Веса для разных типов моделей
WEIGHTS = {
    "cuad": 4.0,           # CUAD-специализированные самые важные
    "primary": 3.0,        # Основные SQuAD модели
    "fallback": 2.0,       # Запасные варианты
}

# Эпохи для разных моделей
EPOCHS = {
    "deberta": 5,
    "large": 4,
    "base": 4,
    "distilbert": 3,
}

# Batch size (large модели требуют меньше)
BATCH_SIZE = {
    "large": 4,
    "base": 8,
    "distilbert": 16,
    "albert-xxlarge": 2,
}

for i, model_name in enumerate(final_models, 1):
    # Определяем тип модели
    if any(c in model_name.lower() for c in ["cuad"]):
        model_type = "cuad"
    elif any(c in model_name.lower() for c in ["deberta", "roberta-large", "albert-xxlarge"]):
        model_type = "primary"
    else:
        model_type = "fallback"

    # Определяем batch size
    batch = 4
    if "base" in model_name.lower():
        batch = 8
    elif "distilbert" in model_name.lower():
        batch = 16
    elif "large" in model_name.lower():
        batch = 4
    elif "albert-xxlarge" in model_name.lower():
        batch = 2

    # Определяем learning rate
    lr = 1e-5 if "large" in model_name.lower() or "xxlarge" in model_name.lower() else 2e-5

    # Определяем эпохи
    epochs = 4
    if "deberta" in model_name.lower():
        epochs = 5
    elif "distilbert" in model_name.lower():
        epochs = 3

    MODELS_CONFIG.append({
        "checkpoint": model_name,
        "lr": lr,
        "epochs": epochs,
        "batch": batch,
        "weight": WEIGHTS.get(model_type, 2.5),
        "swa": 2,
        "type": model_type,
    })

    print(f"  {i}. {model_name}")
    print(f"     lr={lr}, epochs={epochs}, batch={batch}, weight={WEIGHTS.get(model_type, 2.5)}")

# Сохраняем конфиг для основной ячейки
with open('models_config.json', 'w') as f:
    json.dump(MODELS_CONFIG, f, indent=2)

print("\n" + "="*70)
print("✓ ПРОВЕРКА ЗАВЕРШЕНА. КОНФИГ СОХРАНЁН В models_config.json")
print("  Теперь можно запускать ЯЧЕЙКУ 2 (основной код)")
print("="*70)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА 2: ОСНОВНОЙ КОД — АНСАМБЛЬ МОДЕЛЕЙ ДЛЯ CUAD
# ──────────────────────────────────────────────────────────────────────────────

import gc
import copy
import json
import collections
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    TrainingArguments, Trainer, DefaultDataCollator,
    TrainerCallback,
)
from datasets import Dataset

# Очищаем память
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Константы
SEED = 42
MAX_LENGTH = 384
MAX_ANS_LEN = 1000
N_BEST = 20
TTA_STRIDES = [64, 128, 192]
SWA_LAST_N = 2

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1

print("="*70)
print("ОСНОВНОЙ КОД: АНСАМБЛЬ МОДЕЛЕЙ ДЛЯ CUAD")
print("="*70)
print(f"Device: {'GPU ✓' if DEVICE == 0 else 'CPU'}")
if torch.cuda.is_available():
    free = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU RAM: {free:.1f}/{total:.1f} GB свободно")

# Загружаем конфиг моделей из проверки
try:
    with open('models_config.json', 'r') as f:
        MODELS_CONFIG = json.load(f)
    print(f"\n✓ Загружено {len(MODELS_CONFIG)} моделей из конфига:")
    for cfg in MODELS_CONFIG:
        print(f"  - {cfg['checkpoint']} (type={cfg['type']}, weight={cfg['weight']})")
except FileNotFoundError:
    print("\n❌ Ошибка: models_config.json не найден!")
    print("  Сначала выполните ЯЧЕЙКУ 1 (проверка моделей)")
    raise

# ── ЗАГРУЗКА ДАННЫХ ─────────────────────────────────────────────
print("\n" + "="*70)
print("ЗАГРУЗКА ДАННЫХ")
print("="*70)

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sub_df = pd.read_csv("sample_submission.csv")

def build_answers_col(df):
    df = df.copy()
    df["answers"] = df.apply(
        lambda r: {"text": [r["answers"]], "answer_start": [r["answer_start"]]},
        axis=1,
    )
    return df

train_df = build_answers_col(train_df)
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
print(f"Train: {len(train_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

# ── SWA CALLBACK (усреднение весов) ────────────────────────────
class SWACallback(TrainerCallback):
    def __init__(self, last_n=2):
        self.last_n = last_n
        self.checkpoints = []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        self.checkpoints.append(copy.deepcopy(model.state_dict()))
        if len(self.checkpoints) > self.last_n:
            dropped = self.checkpoints.pop(0)
            del dropped
            gc.collect()
        print(f"    [SWA] эпоха {state.epoch:.0f} | буфер: {len(self.checkpoints)}/{self.last_n}")

def average_weights(model, checkpoints):
    if not checkpoints:
        return model
    avg = {}
    for key in checkpoints[0]:
        stacked = torch.stack([c[key].float() for c in checkpoints])
        avg[key] = stacked.mean(0).to(checkpoints[0][key].dtype)
    model.load_state_dict(avg)
    print(f"    [SWA] усреднено {len(checkpoints)} чекпоинтов")
    return model

# ── ПОСТОБРАБОТКА ТЕКСТА ───────────────────────────────────────
def clean_span(text: str) -> str:
    if not text:
        return text
    text = text.strip()
    for o, c in [("(", ")"), ("[", "]"), ("{", "}")]:
        while text.endswith(o) and text.count(o) > text.count(c):
            text = text[:-1].strip()
        while text.startswith(c) and text.count(c) > text.count(o):
            text = text[1:].strip()
    while text and text[0] in ",; ":
        text = text[1:]
    return text.strip()

# ── ТОКЕНИЗАЦИЯ ДЛЯ ОБУЧЕНИЯ ───────────────────────────────────
def make_train_features(tokenizer, stride=128):
    def _fn(examples):
        tok = tokenizer(
            examples["question"], examples["context"],
            truncation="only_second", max_length=MAX_LENGTH,
            stride=stride, return_overflowing_tokens=True,
            return_offsets_mapping=True, padding="max_length",
        )
        sample_map = tok.pop("overflow_to_sample_mapping")
        offset_map = tok.pop("offset_mapping")
        starts, ends = [], []

        for i, offsets in enumerate(offset_map):
            input_ids = tok["input_ids"][i]
            cls_idx = input_ids.index(tokenizer.cls_token_id)
            seq_ids = tok.sequence_ids(i)
            answers = examples["answers"][sample_map[i]]

            if len(answers["answer_start"]) == 0:
                starts.append(cls_idx)
                ends.append(cls_idx)
                continue

            sc = answers["answer_start"][0]
            ec = sc + len(answers["text"][0])

            ts = next(j for j, s in enumerate(seq_ids) if s == 1)
            te = len(seq_ids) - 1
            while seq_ids[te] != 1:
                te -= 1

            if not (offsets[ts][0] <= sc <= offsets[te][1]):
                starts.append(cls_idx)
                ends.append(cls_idx)
                continue

            s = ts
            while s < len(offsets) and offsets[s][0] <= sc:
                s += 1
            starts.append(s - 1)

            e = te
            while e >= ts and offsets[e][1] >= ec:
                e -= 1
            ends.append(e + 1)

        tok["start_positions"] = starts
        tok["end_positions"] = ends
        return tok
    return _fn

# ── ТОКЕНИЗАЦИЯ ДЛЯ ИНФЕРЕНСА ──────────────────────────────────
def make_val_features(tokenizer, stride=128):
    def _fn(examples):
        tok = tokenizer(
            examples["question"], examples["context"],
            truncation="only_second", max_length=MAX_LENGTH,
            stride=stride, return_overflowing_tokens=True,
            return_offsets_mapping=True, padding="max_length",
        )
        sample_map = tok.pop("overflow_to_sample_mapping")
        tok["example_id"] = []
        for i in range(len(tok["input_ids"])):
            seq_ids = tok.sequence_ids(i)
            sample_idx = sample_map[i]
            tok["example_id"].append(examples["id"][sample_idx])
            tok["offset_mapping"][i] = [
                (o if seq_ids[k] == 1 else None)
                for k, o in enumerate(tok["offset_mapping"][i])
            ]
        return tok
    return _fn

# ── TTA ИНФЕРЕНС ───────────────────────────────────────────────
def predict_with_tta(model, tokenizer, strides=TTA_STRIDES):
    device = next(model.parameters()).device
    id2idx = {k: i for i, k in enumerate(test_dataset["id"])}
    all_cands = collections.defaultdict(list)
    model.eval()

    for stride in strides:
        print(f"    [TTA] stride={stride}...", end=" ", flush=True)

        tokenized_test = test_dataset.map(
            make_val_features(tokenizer, stride=stride),
            batched=True,
            remove_columns=test_dataset.column_names,
        )
        ids_all = tokenized_test["input_ids"]
        attn_all = tokenized_test["attention_mask"]
        all_sl, all_el = [], []

        with torch.no_grad():
            batch_size = 8  # фиксированный batch для stability
            for b in range(0, len(ids_all), batch_size):
                inp = torch.tensor(ids_all[b:b+batch_size]).to(device)
                attn = torch.tensor(attn_all[b:b+batch_size]).to(device)
                out = model(input_ids=inp, attention_mask=attn)
                all_sl.append(out.start_logits.cpu().numpy())
                all_el.append(out.end_logits.cpu().numpy())
                del inp, attn, out

        sl = np.concatenate(all_sl, 0)
        el = np.concatenate(all_el, 0)

        feat_per_ex = collections.defaultdict(list)
        for i, feat in enumerate(tokenized_test):
            feat_per_ex[id2idx[feat["example_id"]]].append(i)

        for ex_idx, example in enumerate(test_dataset):
            ctx = example["context"]
            for fi in feat_per_ex[ex_idx]:
                of = tokenized_test[fi]["offset_mapping"]
                sli = sl[fi]
                eli = el[fi]

                top_starts = np.argsort(sli)[-1:-N_BEST-1:-1].tolist()
                top_ends = np.argsort(eli)[-1:-N_BEST-1:-1].tolist()

                for si in top_starts:
                    for ei in top_ends:
                        if si >= len(of) or ei >= len(of):
                            continue
                        if of[si] is None or of[ei] is None:
                            continue
                        if ei < si or ei - si + 1 > MAX_ANS_LEN:
                            continue

                        text = clean_span(ctx[of[si][0]: of[ei][1]])
                        if not text:
                            continue

                        all_cands[example["id"]].append({
                            "score": float(sli[si]) + float(eli[ei]),
                            "text": text,
                        })

        del tokenized_test, ids_all, attn_all, all_sl, all_el, sl, el
        gc.collect()
        print("ok")

    result = {}
    for qid in [ex["id"] for ex in test_dataset]:
        cands = all_cands.get(qid, [])
        if cands:
            best = max(cands, key=lambda x: x["score"])
            result[qid] = (best["text"], best["score"])
        else:
            result[qid] = ("", -1e9)
    return result

# ── ОБУЧЕНИЕ ОДНОЙ МОДЕЛИ ──────────────────────────────────────
def train_one(checkpoint, lr, epochs, batch, swa_last_n=SWA_LAST_N, warmup=0.1):
    print(f"\n{'='*65}")
    print(f"  МОДЕЛЬ: {checkpoint}")
    print(f"  lr={lr} | epochs={epochs} | batch={batch} | SWA_last={swa_last_n}")
    print('='*65)

    try:
        tok = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)
    except Exception as e:
        print(f"  ✗ Ошибка загрузки: {e}")
        return None, None

    # Включаем gradient checkpointing для экономии памяти
    model.gradient_checkpointing_enable()

    # Токенизируем тренировочные данные
    tokenized = train_dataset.map(
        make_train_features(tok),
        batched=True,
        remove_columns=train_dataset.column_names,
    )

    swa_cb = SWACallback(last_n=swa_last_n)

    args = TrainingArguments(
        output_dir=f"./ckpt_{checkpoint.replace('/','_')}",
        learning_rate=lr,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=2,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=warmup,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        save_strategy="no",
        seed=SEED,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized,
        data_collator=DefaultDataCollator(),
        callbacks=[swa_cb],
    )

    trainer.train()

    # Освобождаем память
    del trainer, tokenized
    gc.collect()
    torch.cuda.empty_cache()

    # Усредняем веса (SWA)
    if len(swa_cb.checkpoints) >= 2:
        model = average_weights(model, swa_cb.checkpoints)

    del swa_cb
    gc.collect()

    model.eval()
    return model, tok

# ── RANK-НОРМАЛИЗАЦИЯ ДЛЯ АНСАМБЛЯ ────────────────────────────
def rank_normalize(all_preds):
    all_ids = list(all_preds[0].keys())
    normalized = []
    for preds in all_preds:
        scores = np.array([preds[qid][1] for qid in all_ids])
        ranks = np.argsort(np.argsort(scores)).astype(float)
        if ranks.max() > 0:
            ranks /= ranks.max()
        normalized.append({
            qid: (preds[qid][0], float(ranks[i]))
            for i, qid in enumerate(all_ids)
        })
    return normalized

def ensemble_final(all_preds, weights):
    """
    Ансамблирование с rank-нормализацией и бонусом за консенсус
    """
    normalized = rank_normalize(all_preds)
    final = {}

    for qid in all_preds[0]:
        scores = collections.defaultdict(float)
        votes = collections.Counter()

        for norm_preds, w in zip(normalized, weights):
            text, norm_score = norm_preds[qid]
            if not text.strip():
                continue
            scores[text] += w * norm_score
            votes[text] += 1

        # Бонус за консенсус
        for text, v in votes.items():
            if v >= 3:
                scores[text] *= 2.0
            elif v >= 2:
                scores[text] *= 1.5

        if scores:
            final[qid] = max(scores, key=scores.get)
        else:
            # Fallback: берём лучший по сырой вероятности
            best_text, best_score = "", -1e9
            for preds, _ in zip(all_preds, weights):
                text, score = preds[qid]
                if score > best_score and text.strip():
                    best_text, best_score = text, score
            final[qid] = best_text

    return final

# ── ОСНОВНОЙ ЦИКЛ ОБУЧЕНИЯ ─────────────────────────────────────
print("\n" + "="*70)
print("НАЧАЛО ОБУЧЕНИЯ АНСАМБЛЯ")
print("="*70)

all_preds = []
all_weights = []
ok_names = []

for i, cfg in enumerate(MODELS_CONFIG, 1):
    print(f"\n[{i}/{len(MODELS_CONFIG)}] Обработка модели...")

    # Очищаем память перед каждой моделью
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"  GPU свободно: {free:.1f} GB")

    # Обучение
    model, tok = train_one(
        cfg["checkpoint"],
        cfg["lr"],
        cfg["epochs"],
        cfg["batch"],
        swa_last_n=cfg["swa"],
    )

    if model is None:
        print(f"  ✗ Модель {cfg['checkpoint']} пропущена")
        continue

    # Инференс с TTA
    print(f"  → TTA инференс (strides={TTA_STRIDES})...")
    preds = predict_with_tta(model, tok, strides=TTA_STRIDES)

    all_preds.append(preds)
    all_weights.append(cfg["weight"])
    ok_names.append(cfg["checkpoint"])

    # Освобождаем память
    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"  ✓ Модель {i}/{len(MODELS_CONFIG)} завершена")

# Проверка, что хотя бы одна модель обучилась
if not all_preds:
    raise RuntimeError("❌ Ни одна модель не обучилась успешно!")

print(f"\n✓ Успешно обучено: {len(all_preds)}/{len(MODELS_CONFIG)} моделей")

# ── АНСАМБЛИРОВАНИЕ ───────────────────────────────────────────
print("\n" + "="*70)
print("ФОРМИРОВАНИЕ ФИНАЛЬНОГО АНСАМБЛЯ")
print("="*70)

if len(all_preds) > 1:
    final_preds = ensemble_final(all_preds, all_weights)

    # Статистика согласия моделей
    agree_all = agree_2 = alone = 0
    for qid in [ex["id"] for ex in test_dataset]:
        texts = [p[qid][0] for p in all_preds]
        top_votes = max(collections.Counter(texts).values())
        if top_votes == len(all_preds):
            agree_all += 1
        elif top_votes >= 2:
            agree_2 += 1
        else:
            alone += 1

    print(f"\n  Статистика ансамбля:")
    print(f"    Все {len(all_preds)} модели согласны: {agree_all}")
    print(f"    Хотя бы 2 согласны: {agree_2}")
    print(f"    Полное расхождение: {alone}")
else:
    final_preds = {qid: text for qid, (text, _) in all_preds[0].items()}
    print("  Используется одна модель (ансамбль не сформирован)")

# ── СОХРАНЕНИЕ РЕЗУЛЬТАТОВ ─────────────────────────────────────
print("\n" + "="*70)
print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*70)

# Финальный сабмишн
submission = sub_df[["id"]].copy()
submission["answers"] = submission["id"].map(final_preds).fillna("")
submission.to_csv("submission.csv", index=False)

# Детали по каждой модели
details = sub_df[["id"]].copy()
for i, (preds, name) in enumerate(zip(all_preds, ok_names), 1):
    short_name = name.split('/')[-1][:20]
    col = f"model_{i}_{short_name}"
    details[col] = details["id"].map({qid: text for qid, (text, _) in preds.items()}).fillna("")

details["final_ensemble"] = details["id"].map(final_preds).fillna("")
details.to_csv("submission_details.csv", index=False)

# Пустые ответы
empty_count = (submission['answers'] == '').sum()
print(f"\n  submission.csv — финальные предсказания")
print(f"  submission_details.csv — предсказания каждой модели")
print(f"  Пустых ответов: {empty_count}/{len(submission)} ({100*empty_count/len(submission):.1f}%)")

print("\n" + "="*70)
print("ГОТОВО!")
print("="*70)

# Показываем первые 10 строк
print("\nПример предсказаний:")
print(submission.head(10).to_string())

# Для скачивания в Colab (раскомментировать при необходимости)
# from google.colab import files
# files.download("submission.csv")
# files.download("submission_details.csv")